# Diffusion Policy sur PushT — LeRobot (Colab GPU)

Entraînement d'une **Diffusion Policy** sur le dataset `lerobot/pusht` avec tous les paramètres par défaut de LeRobot.

**Avant de lancer** : `Runtime → Change runtime type → T4 GPU`.

Temps estimé : ~3h sur T4 (200k steps, valeur par défaut).
Performance attendue : ~84-91% success rate (Chi et al. 2023).

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi

## 2. Installer LeRobot

On installe avec le sous-ensemble `pusht` pour avoir `gym_pusht` inclus.

In [ ]:
!pip install -q 'lerobot[pusht]'

## 3. Entraînement

Tous les paramètres par défaut :
- Policy : `diffusion` (Diffusion Policy CNN-UNet 1D)
- Dataset : `lerobot/pusht`
- Steps : 200 000 (défaut LeRobot Diffusion Policy)
- Batch size : 64
- Optimizer : AdamW, LR 1e-4
- Eval automatique pendant le training
- Checkpoints sauvegardés dans `outputs/train/diffusion_pusht/checkpoints/`

On désactive `push_to_hub` pour rester 100% local (sinon LeRobot exige `policy.repo_id`).

In [ ]:
!lerobot-train \
  --dataset.repo_id=lerobot/pusht \
  --policy.type=diffusion \
  --policy.push_to_hub=false \
  --env.type=pusht \
  --output_dir=outputs/train/diffusion_pusht \
  --job_name=diffusion_pusht \
  --policy.device=cuda

## 4. Vérifier les checkpoints produits

Utile avant de lancer l'éval — confirme que le path existe bien.

In [ ]:
!ls -la outputs/train/diffusion_pusht/checkpoints/

## 5. Évaluation (50 épisodes)

Charge le dernier checkpoint et évalue sur 50 épisodes PushT.

In [ ]:
!lerobot-eval \
  --policy.path=outputs/train/diffusion_pusht/checkpoints/last/pretrained_model \
  --env.type=pusht \
  --eval.n_episodes=50 \
  --eval.batch_size=1

## 6. Récupérer le checkpoint

Zippe le dossier du modèle entraîné pour pouvoir le télécharger localement.

In [ ]:
import shutil
shutil.make_archive('diffusion_pusht_checkpoint', 'zip', 'outputs/train/diffusion_pusht/checkpoints/last')
print('Archive prête : diffusion_pusht_checkpoint.zip')

In [ ]:
from google.colab import files
files.download('diffusion_pusht_checkpoint.zip')